# Chapter 11: Embeddings — Numbers to Meaning Vectors

After tokenization, each token is an integer ID. But IDs have no meaning:

```
"cat" = 42,  "dog" = 87,  "the" = 1
```

Is cat closer to dog or to the? The numbers don't tell us.

**Embeddings** convert each ID into a **vector** (list of numbers) where **meaning is encoded in position**:

```
"cat" → [0.8, 0.2, -0.1, 0.9]   ← animal-like, small, ...
"dog" → [0.7, 0.3, -0.2, 0.8]   ← similar to cat!
"the" → [0.0, 0.0,  0.5, 0.1]   ← very different
```

Similar words end up **close together** in this vector space.

---
## Embedding Table: Just a Lookup

An embedding is simply a **table** where each row is a token's vector.

```
ID 0 ("the") → [0.01, 0.05, 0.50, 0.10]
ID 1 ("cat") → [0.80, 0.20, -0.10, 0.90]
ID 2 ("dog") → [0.70, 0.30, -0.20, 0.80]
...
```

The vectors start random and are **learned during training** — the model discovers what each dimension should mean.

In [ ]:
import numpy as np
np.random.seed(42)

# Simple embedding table
vocab = {"the": 0, "cat": 1, "dog": 2, "sat": 3, "ran": 4, 
         "king": 5, "queen": 6, "man": 7, "woman": 8, "happy": 9}
vocab_size = len(vocab)
embedding_dim = 4  # real models use 768-4096

# Initialize randomly (before training)
embedding_table = np.random.randn(vocab_size, embedding_dim) * 0.3

print("Embedding Table (before training — random):\n")
print(f"  {'Token':<8} {'ID':>3}   {'Vector (4 dimensions)'}")
print(f"  {'─'*8} {'─'*3}   {'─'*35}")
for word, idx in vocab.items():
    vec = embedding_table[idx]
    vec_str = ', '.join(f'{v:+.3f}' for v in vec)
    print(f"  {word:<8} {idx:>3}   [{vec_str}]")

print("\nLookup: just index into the table.")
print(f"  embed('cat') = embedding_table[1] = {embedding_table[1].round(3)}")

In [ ]:
# Embedding a sentence
sentence = "the cat sat"
token_ids = [vocab[w] for w in sentence.split()]

print(f"Embedding a sentence:\n")
print(f"  Text:   '{sentence}'")
print(f"  IDs:    {token_ids}")
print(f"  Shapes: each ID → {embedding_dim}-dim vector\n")

embedded = embedding_table[token_ids]  # batch lookup
print(f"  Result: {len(token_ids)} tokens × {embedding_dim} dimensions = shape {embedded.shape}\n")
for word, vec in zip(sentence.split(), embedded):
    print(f"  '{word}' → [{', '.join(f'{v:+.3f}' for v in vec)}]")

print(f"\nThis matrix goes into the transformer (Ch 12).")

---
## How Embeddings Learn Meaning

During training, words that appear in **similar contexts** get pushed toward similar vectors.

```
"The cat sat on the mat"      ← cat appears near sat, mat
"The dog sat on the rug"      ← dog appears near sat, rug
→ cat and dog get similar vectors (same context patterns)
```

This is the **distributional hypothesis**: "You shall know a word by the company it keeps."

In [ ]:
# Simulate trained embeddings to show learned relationships
# These are hand-crafted to demonstrate the concept

trained_embeddings = {
    # [royalty, gender, animal, action]
    "king":   np.array([ 0.9,  0.8,  0.0,  0.1]),
    "queen":  np.array([ 0.9, -0.8,  0.0,  0.1]),
    "man":    np.array([ 0.1,  0.8,  0.0,  0.2]),
    "woman":  np.array([ 0.1, -0.8,  0.0,  0.2]),
    "cat":    np.array([ 0.0,  0.0,  0.9,  0.3]),
    "dog":    np.array([ 0.0,  0.1,  0.8,  0.4]),
    "sat":    np.array([ 0.0,  0.0,  0.1,  0.9]),
    "ran":    np.array([ 0.0,  0.0,  0.1,  0.8]),
    "happy":  np.array([ 0.0,  0.0,  0.0,  0.5]),
}

print("Trained Embeddings (after learning from text):\n")
print(f"  {'Word':<8} {'[royalty':>9} {'gender':>8} {'animal':>8} {'action]':>8}")
print(f"  {'─'*8} {'─'*9} {'─'*8} {'─'*8} {'─'*8}")
for word, vec in trained_embeddings.items():
    print(f"  {word:<8} {vec[0]:>+9.1f} {vec[1]:>+8.1f} {vec[2]:>+8.1f} {vec[3]:>+8.1f}")

print("\nNotice:")
print("  king & queen: same royalty, opposite gender")
print("  cat & dog: same animal dimension, different from king/queen")
print("  sat & ran: same action type, different from animals")

---
## Similarity: Cosine Distance

How do we measure if two vectors are "close"?

**Cosine similarity**: measures the angle between two vectors.
- **1.0** = identical direction (same meaning)
- **0.0** = perpendicular (unrelated)
- **-1.0** = opposite direction (opposite meaning)

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Compare all pairs
words = list(trained_embeddings.keys())
vecs = list(trained_embeddings.values())

print("Cosine Similarity Between Word Pairs:\n")

interesting_pairs = [
    ("cat", "dog",    "both animals"),
    ("king", "queen", "both royalty"),
    ("man", "woman",  "both people"),
    ("sat", "ran",    "both actions"),
    ("cat", "king",   "unrelated"),
    ("dog", "happy",  "unrelated"),
    ("king", "man",   "both male"),
    ("queen", "woman", "both female"),
]

print(f"  {'Word 1':<8} {'Word 2':<8} {'Similarity':>10} {'Why':>20}")
print(f"  {'─'*8} {'─'*8} {'─'*10} {'─'*20}")

for w1, w2, reason in interesting_pairs:
    sim = cosine_similarity(trained_embeddings[w1], trained_embeddings[w2])
    bar_len = int((sim + 1) / 2 * 15)  # -1 to 1 → 0 to 15
    bar = "█" * bar_len
    print(f"  {w1:<8} {w2:<8} {sim:>+10.3f} {reason:>20}  {bar}")

---
## The Famous Example: Vector Arithmetic

The most famous embedding property:

```
king - man + woman ≈ queen
```

This works because embeddings capture **relationships as directions**:
- `king - man` = the "royalty" direction
- Adding that to `woman` = `queen`

In [ ]:
# Vector arithmetic
king = trained_embeddings["king"]
man = trained_embeddings["man"]
woman = trained_embeddings["woman"]
queen = trained_embeddings["queen"]

result = king - man + woman

print("Vector Arithmetic: king - man + woman = ?\n")
print(f"  king:           {king}")
print(f"  - man:          {man}")
print(f"  + woman:        {woman}")
print(f"  ─────────────────────────")
print(f"  = result:       {result}")
print(f"  actual queen:   {queen}")
print(f"\n  Similarity of result to queen: {cosine_similarity(result, queen):.3f}")

# Find the closest word to the result
print("\n  Closest words to result:")
similarities = []
for word, vec in trained_embeddings.items():
    sim = cosine_similarity(result, vec)
    similarities.append((word, sim))
similarities.sort(key=lambda x: -x[1])
for word, sim in similarities[:5]:
    marker = " ← closest!" if word == similarities[0][0] else ""
    print(f"    {word:<8} {sim:>+.3f}{marker}")

---
## Positional Encoding: Where Are You in the Sentence?

Embeddings alone don't capture **word order**:

```
"dog bites man" and "man bites dog" would have the same embeddings!
```

Solution: **add position information** to each embedding.

```
final_embedding = token_embedding + position_embedding
```

Transformers use sinusoidal patterns so the model can learn relative positions:

In [ ]:
def positional_encoding(seq_len, d_model):
    """Sinusoidal positional encoding (from 'Attention Is All You Need')."""
    pe = np.zeros((seq_len, d_model))
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            pe[pos, i] = np.sin(pos / (10000 ** (i / d_model)))
            if i + 1 < d_model:
                pe[pos, i+1] = np.cos(pos / (10000 ** (i / d_model)))
    return pe

# Show positional encoding
seq_len = 6
d_model = 8
pe = positional_encoding(seq_len, d_model)

sentence = ["The", "cat", "sat", "on", "the", "mat"]

print("Positional Encoding (sinusoidal pattern):\n")
print(f"  {'Pos':<4} {'Word':<6}", end="")
for i in range(d_model):
    print(f" dim{i:>2}", end="")
print()
print(f"  {'─'*4} {'─'*6}", end="")
print(f" {'─'*5}" * d_model)

for pos in range(seq_len):
    print(f"  {pos:<4} {sentence[pos]:<6}", end="")
    for i in range(d_model):
        print(f" {pe[pos, i]:>+5.2f}", end="")
    print()

print("\nEach position gets a unique pattern.")
print("Nearby positions have similar patterns (smooth transitions).")
print("This lets the model understand 'word 3 is near word 4'.")

In [ ]:
# Show how position changes the final embedding
print("Final Embedding = Token Embedding + Position Encoding:\n")

# Use simple embeddings for demo
token_embs = np.random.randn(seq_len, d_model) * 0.3

word = "the"
pos0 = token_embs[0] + pe[0]  # "the" at position 0
pos4 = token_embs[4] + pe[4]  # "the" at position 4

print(f"  'the' at position 0: [{', '.join(f'{v:+.2f}' for v in pos0[:4])}  ...]")
print(f"  'the' at position 4: [{', '.join(f'{v:+.2f}' for v in pos4[:4])}  ...]")
print(f"\n  Same word, different positions → different vectors!")
print(f"  Similarity: {cosine_similarity(pos0, pos4):.3f}")
print(f"  (Similar but not identical — position matters)")

---
## Embedding Dimensions in Real Models

```
Word2Vec (2013):     300 dimensions
BERT (2018):         768 dimensions
GPT-3 (2020):       12,288 dimensions
GPT-4 / Claude:     ~8,000-12,000 dimensions (estimated)
```

More dimensions = can capture more nuances of meaning, but costs more memory and computation.

In [ ]:
# Embedding table sizes in real models
print("Embedding Table Sizes in Real Models:\n")
print(f"  {'Model':<16} {'Vocab':>8} {'Dim':>6} {'Table Size':>14} {'Note'}")
print(f"  {'─'*16} {'─'*8} {'─'*6} {'─'*14} {'─'*20}")

models = [
    ("Word2Vec",     50000,   300,  "just embeddings"),
    ("BERT-base",    30522,   768,  "110M total params"),
    ("GPT-2",        50257,   768,  "117M total params"),
    ("GPT-3",       50257,  12288, "175B total params"),
    ("LLaMA-7B",    32000,   4096, "7B total params"),
]

for name, vocab_sz, dim, note in models:
    table_params = vocab_sz * dim
    if table_params > 1e9:
        size_str = f"{table_params/1e9:.1f}B"
    elif table_params > 1e6:
        size_str = f"{table_params/1e6:.0f}M"
    else:
        size_str = f"{table_params/1e3:.0f}K"
    print(f"  {name:<16} {vocab_sz:>8,} {dim:>6} {size_str:>14} {note}")

print("\nThe embedding table is a small fraction of total parameters.")
print("Most parameters are in the transformer layers (Ch 12).")

---
## Summary

| Concept | Key Point |
|---------|----------|
| **Embedding** | Lookup table: token ID → dense vector |
| **Learning** | Vectors are learned during training, not hand-coded |
| **Similarity** | Similar words → similar vectors (cosine similarity) |
| **Arithmetic** | king - man + woman ≈ queen |
| **Positional encoding** | Added to embeddings so model knows word order |
| **Dimensions** | 300 (Word2Vec) to 12,000+ (GPT-3) |

**Next chapter**: Attention & Transformers — how these vectors interact with each other